# SMS Spam Classifier — Kaggle Notebook
Datasets: `tinu10kumar/sms-spam-dataset`, `gevabriel/indonesian-sms-spam`

This notebook mirrors the local pipeline: downloads both corpora (if added via Notebook Settings), merges them, runs TF-IDF + Logistic Regression hyperparameter search, and exports JSON artifacts.

In [ ]:
import json, re, time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, f1_score

CLEAN_URL = re.compile(r"https?://\S+")
CLEAN_NUM = re.compile(r"\b\d+\b")
CLEAN_CURRENCY = re.compile(r"[$€£¥]+")
CLEAN_MULTI_SPACE = re.compile(r"\s+")
DATASET_DIRS = [
    Path('/kaggle/input/sms-spam-dataset'),
    Path('/kaggle/input/indonesian-sms-spam'),
]


def clean_text(text):
    text = (text or "").lower()
    text = CLEAN_URL.sub(" url ", text)
    text = CLEAN_CURRENCY.sub(" currency ", text)
    text = CLEAN_NUM.sub(" num ", text)
    return CLEAN_MULTI_SPACE.sub(" ", text).strip()


def discover_file(base_dir: Path) -> Path:
    candidates = [p for p in base_dir.rglob('*') if p.is_file() and p.suffix.lower() in {'.csv', '.tsv', '.txt'}]
    if not candidates:
        raise FileNotFoundError(f'No tabular file found in {base_dir}')
    candidates.sort(key=lambda p: (len(p.parts), p.name))
    return candidates[0]


def read_table(path: Path) -> pd.DataFrame:
    attempts = [
        ('utf-8', {'sep': ','}),
        ('utf-8', {'sep': None, 'engine': 'python'}),
        ('latin1', {'sep': ','}),
        ('latin1', {'sep': None, 'engine': 'python'}),
    ]
    for encoding, kwargs in attempts:
        try:
            return pd.read_csv(path, encoding=encoding, **kwargs)
        except Exception:
            continue
    raise ValueError(f'Unable to parse {path}')


def normalise(df: pd.DataFrame) -> pd.DataFrame:
    cols = [str(c).strip().lower() for c in df.columns]
    df.columns = cols
    label_candidates = [c for c in cols if any(tok in c for tok in ['label', 'class', 'category', 'status', 'spam'])]
    text_candidates = [c for c in cols if any(tok in c for tok in ['text', 'message', 'sms', 'content', 'body'])]
    label_col = label_candidates[0] if label_candidates else cols[0]
    text_col = text_candidates[0] if text_candidates else (cols[1] if len(cols) > 1 else cols[0])
    slim = df[[label_col, text_col]].rename(columns={label_col: 'label', text_col: 'text'})
    slim['label'] = slim['label'].map(lambda val: 'spam' if 'spam' in str(val).lower() else 'ham')
    slim['text'] = slim['text'].astype(str)
    return slim.dropna(subset=['label', 'text'])

frames = []
for dataset_dir in DATASET_DIRS:
    if not dataset_dir.exists():
        print(f'Skipping {dataset_dir}: not added to notebook data sources.')
        continue
    data_file = discover_file(dataset_dir)
    df_raw = read_table(data_file)
    df = normalise(df_raw)
    df['source'] = dataset_dir.name
    frames.append(df)
    print(f"Loaded {df.shape[0]} rows from {dataset_dir.name}")

if not frames:
    raise SystemExit('No datasets found. Add them via Kaggle notebook settings.')

df = pd.concat(frames, ignore_index=True)
print('Combined shape before dedupe:', df.shape)
df = df.drop_duplicates(subset='text')
print('Combined shape after dedupe:', df.shape)

texts = df['text'].map(clean_text).to_numpy()
labels = (df['label'] == 'spam').astype(int).to_numpy()

param_grid = [
    {'ngram_range': (1, 2), 'min_df': 1, 'max_df': 0.98, 'max_features': 20000, 'C': 1.5},
    {'ngram_range': (1, 3), 'min_df': 1, 'max_df': 0.97, 'max_features': 30000, 'C': 2.0},
    {'ngram_range': (1, 4), 'min_df': 1, 'max_df': 0.96, 'max_features': 32000, 'C': 2.5},
    {'ngram_range': (1, 4), 'min_df': 2, 'max_df': 0.95, 'max_features': 28000, 'C': 3.0},
]


def cv_score(params, seed=42):
    splitter = StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
    scores = []
    for train_idx, valid_idx in splitter.split(texts, labels):
        vect = TfidfVectorizer(
            ngram_range=params['ngram_range'],
            min_df=params['min_df'],
            max_df=params['max_df'],
            max_features=params['max_features'],
            sublinear_tf=True,
            norm='l2',
            strip_accents='unicode',
            lowercase=True,
            token_pattern=r"[a-z']+",
        )
        X_train = vect.fit_transform(texts[train_idx])
        X_valid = vect.transform(texts[valid_idx])
        clf = LogisticRegression(
            max_iter=400, C=params['C'], solver='liblinear', class_weight='balanced', random_state=seed
        )
        clf.fit(X_train, labels[train_idx])
        scores.append(f1_score(labels[valid_idx], clf.predict(X_valid)))
    return float(np.mean(scores))

best = max(param_grid, key=lambda p: cv_score(p))
best_C = best.pop('C')
print('Best params:', best, 'C=', best_C)

vectorizer = TfidfVectorizer(
    ngram_range=best['ngram_range'],
    min_df=best['min_df'],
    max_df=best['max_df'],
    max_features=best['max_features'],
    sublinear_tf=True,
    norm='l2',
    strip_accents='unicode',
    lowercase=True,
    token_pattern=r"[a-z']+",
)
X_train, X_valid, y_train, y_valid = train_test_split(
    texts, labels, test_size=0.2, random_state=42, stratify=labels
)
X_train_vec = vectorizer.fit_transform(X_train)
X_valid_vec = vectorizer.transform(X_valid)
model = LogisticRegression(
    max_iter=400, C=best_C, solver='liblinear', class_weight='balanced', random_state=42
)
model.fit(X_train_vec, y_train)
preds = model.predict(X_valid_vec)
print('Holdout F1', f1_score(y_valid, preds))
print(classification_report(y_valid, preds, digits=4))

model_json = {
    'model_type': 'logreg',
    'version': 'v1.1.0',
    'classes': ['ham', 'spam'],
    'coef': model.coef_.reshape(-1).tolist(),
    'intercept': float(model.intercept_[0]),
    'vocabulary': {tok: int(idx) for tok, idx in vectorizer.vocabulary_.items()},
    'idf': vectorizer.idf_.tolist(),
    'tfidf': {
        'ngram_range': vectorizer.ngram_range,
        'sublinear_tf': True,
        'norm': 'l2',
        'use_idf': True,
        'min_df': vectorizer.min_df,
        'max_df': float(vectorizer.max_df) if hasattr(vectorizer.max_df, '__float__') else vectorizer.max_df,
        'max_features': vectorizer.max_features,
        'lowercase': True,
        'strip_accents': 'unicode',
        'token_pattern': vectorizer.token_pattern,
    },
    'training_meta': {'class_weight': 'balanced', 'solver': 'liblinear', 'max_iter': model.max_iter},
}
for name, payload in [
    ('model.pkl', model_json),
    ('feature_config.json', {'token_pattern': r"[a-zA-Z']+", 'lowercase': True, 'strip': True}),
    ('label_map.json', {'0': 'ham', '1': 'spam'}),
]:
    with open(Path('/kaggle/working') / name, 'w') as fh:
        json.dump(payload, fh, indent=2 if name != 'model.pkl' else None)
print('Saved artifacts to /kaggle/working/*.json')
